### Step 9: Append to Existing Dataset
*Building the Hugging Face hierarchical structure: Dialect/Speaker/Project/parts/*

# Environment & Drive Mount

In [ ]:
import os
import shutil
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
print("\n✅ Environment ready for Hierarchical Dataset Consolidation.")

# The Consolidation Engine

In [ ]:
def get_next_project_number(speaker_dir):
    """Checks the speaker folder and returns the next incremental folder number."""
    if not os.path.exists(speaker_dir):
        return 1

    existing_folders = [f for f in os.listdir(speaker_dir) if os.path.isdir(os.path.join(speaker_dir, f))]
    numbers = []
    for f in existing_folders:
        if f.isdigit():
            numbers.append(int(f))

    return max(numbers) + 1 if numbers else 1

def consolidate_hierarchical(pipeline_workspace, master_dataset_dir):
    master_csv_path = os.path.join(master_dataset_dir, "metadata.csv")
    master_dataframes = []

    print(f"🚀 Starting Hierarchical Consolidation into: {master_dataset_dir}")

    # Load existing master CSV if appending to a previously built dataset
    if os.path.exists(master_csv_path):
        print("📂 Existing master metadata.csv found. Loading to append...")
        master_dataframes.append(pd.read_csv(master_csv_path, sep='|', encoding='utf-8'))

    for sub_dir in os.listdir(pipeline_workspace):
        sub_dir_path = os.path.join(pipeline_workspace, sub_dir)
        if not os.path.isdir(sub_dir_path): continue

        gold_csv_path = os.path.join(sub_dir_path, "Ara170_Gold_Standard.csv")
        parts_dir = os.path.join(sub_dir_path, "Parts")

        if not os.path.exists(gold_csv_path) or not os.path.exists(parts_dir):
            continue

        print(f"\n🛠️ Processing project: {sub_dir}")
        df = pd.read_csv(gold_csv_path, sep='|', encoding='utf-8')

        # Assume 1 speaker/dialect per project folder (based on Notebook 2 logic)
        speaker_id = df.iloc[0]['Speaker_ID']
        dialect = df.iloc[0]['Dialect']

        # Clean dialect string for folder naming (e.g., "Modern Standard Arabic (MSA)" -> "MSA")
        folder_dialect = dialect.split('(')[-1].replace(')', '').strip() if '(' in dialect else dialect.replace(' ', '_')

        # 1. Setup nested hierarchy: Root / Dialect / Speaker / Project_Num
        speaker_dir = os.path.join(master_dataset_dir, folder_dialect, speaker_id)
        os.makedirs(speaker_dir, exist_ok=True)

        project_num = str(get_next_project_number(speaker_dir))
        new_project_dir = os.path.join(speaker_dir, project_num)
        new_parts_dir = os.path.join(new_project_dir, "parts")

        os.makedirs(new_parts_dir, exist_ok=True)

        new_audio_paths = []

        # 2. Copy Audio Files & Generate Relative Paths
        for index, row in df.iterrows():
            clip_num = row['Audio Clip Number']
            old_filepath = os.path.join(parts_dir, f"{clip_num}.wav")
            new_filepath = os.path.join(new_parts_dir, f"{clip_num}.wav")

            if os.path.exists(old_filepath):
                shutil.copy2(old_filepath, new_filepath)
                # Hugging Face relative path format: Dialect/Speaker_ID/Project_Num/parts/1.wav
                relative_path = f"{folder_dialect}/{speaker_id}/{project_num}/parts/{clip_num}.wav"
                new_audio_paths.append(relative_path)
            else:
                print(f"  ⚠️ Missing audio file {old_filepath}. Skipping.")
                new_audio_paths.append(None)

        # 3. Update DataFrame
        df['file_name'] = new_audio_paths
        df = df.dropna(subset=['file_name'])

        # 4. Save Local metadata.csv inside the specific project folder
        local_csv_path = os.path.join(new_project_dir, "metadata.csv")
        df.to_csv(local_csv_path, sep='|', encoding='utf-8', index=False)
        print(f"✅ Created local project folder: {folder_dialect}/{speaker_id}/{project_num}/")

        # Reorder columns for the master dataframe so file_name is first
        cols = ['file_name'] + [c for c in df.columns if c != 'file_name']
        df = df[cols]
        master_dataframes.append(df)

    # 5. Update Master CSV
    if master_dataframes:
        print("\n💾 Updating Master metadata.csv...")
        master_df = pd.concat(master_dataframes, ignore_index=True)

        # Drop duplicates in case a script is run twice accidentally on the same files
        master_df = master_df.drop_duplicates(subset=['file_name'], keep='last')

        master_df.to_csv(master_csv_path, sep='|', encoding='utf-8', index=False)
        print(f"✅ Master CSV updated successfully. Total rows: {len(master_df)}")
    else:
        print("❌ No new data processed.")

# Execution

In [ ]:
pipeline_workspace = input("Enter your Pipeline Workspace directory (from Notebooks 1 & 2): ").strip()
master_dataset_dir = input("Enter the path for the FINAL Master Dataset (e.g., /content/drive/MyDrive/Ara170_Master): ").strip()

consolidate_hierarchical(pipeline_workspace, master_dataset_dir)

# The Resulting Architecture

In [ ]:
Ara170_Master/
├── metadata.csv  (The giant master CSV with relative paths)
├── EGY/
│   ├── SPK_10492/
│   │   ├── 1/
│   │   │   ├── metadata.csv
│   │   │   └── parts/
│   │   │       ├── 1.wav
│   │   │       └── 2.wav
│   │   └── 2/
│   │       ├── metadata.csv
│   │       └── parts/
│   │           ├── 1.wav
│   │           └── 2.wav
├── MSA/
│   └── SPK_88291/
│       └── 1/
│           ├── metadata.csv
│           └── parts/
│               ├── 1.wav